# MSC Geomet - Using Authentication to Access Flow Predictions
# Using the Web Coverage Service

This notebook provides examples of how to use user/password authentication on Meteorological Service of Canada's (MSC) [GeoMet](https://eccc-msc.github.io/open-data/msc-geomet/readme_en/#msc-geomet "MSC GeoMet Open Documentation") platform, in order to access flow prediction through the web coverage service (WCS) with Python. If you are interested in retrieving flow predictions with the web map service (WMS), please see the companion tutorial in this repository. If you are unsure of which service to use, a general rule of thumb is that the web map service is primarily used to retrieve geospatial images and the web coverage service is used to retrieve raw geospatial data. If you are new to the WMS and the WCS we recommend later visiting the [documentation](https://eccc-msc.github.io/open-data/msc-geomet/readme_en/#usage-tutorials-and-technical-documentation "MSC GeoMet Usage, Tutorials, and Technical Documentation") on the MSC Open Data website. If you would like additional details, you can also view the [technical documentation](https://eccc-msc.github.io/open-data/msc-geomet/wcs_en/ "MSC GeoMet Technical Documentation").

## Table of Contents

* [Requirements](#Requirements)
* [Introduction to Web Coverage Service](#Introduction-to-Web-Map-Service)
* [Non-authenticated vs authenticated data](#Non-athenticated-vs-authenticated-data)
* [Accessing layer information](#Accessing-layer-information)
* [Requesting Data with WCS](#Requesting-data-with-WCS)
* [Saving requested data to netCDF or geoJSON](#Saving-requested-data-to-netCDF-or-geoJSON)
* [Bonus: Animating requested data with hvplot](#Bonus:-Visualizing-requested-data-with-hvplot)

## Requirements

First, we import all necessary Python libraries.  Some of these are standard Python libraries, while others are third-party libraries that must first be installed in your Python environment (see environment.yml to create the needed environment). The most important of these third-party libraries is [OWSLib](https://geopython.github.io/OWSLib/index.html "OWSLib Documentation"), which stands for the Open Geospatial Consortium (OGC) web service and which we'll use for accessing the Geomet WMS and WCS. We also recommend [xarray](https://docs.xarray.dev/en/stable/ "xarray Documentation"), for data handling and [cartopy](https://scitools.org.uk/cartopy/docs/latest/) for viewing the data.

In [1]:
# data
import warnings
import re
import time
import configparser
from datetime import datetime, timedelta
import xarray as xr 
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

# web map services 
from owslib.wms import WebMapService
from owslib.wcs import WebCoverageService
from owslib.wcs import Authentication

# plotting
from IPython.display import Image
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
import hvplot.xarray
import panel.widgets as pnw

To access the flow prediction from the user/password authenticated version of GeoMet, we need credentials. We can use the built-in configparser utility library, which reads a file with the following format:

```
[Login]
Username = your_user_name 
Password = your_password
```

In [2]:
config = configparser.ConfigParser()
config.read_file(open('config.cfg'))

login = config['Login']

## Introduction to Web Coverage Service

The GeoMet WCS provides access to the MSC's geospatial data. The WCS provides hydro-meteorological variables from multiple MSC monitoring and numerical modelling systems and their related products.  Included with the data is descriptive information, i.e., metadata.  Together, the descriptive information and images are called layers. 

We can connect to the WCS using OWSlib.  The typical way to do this without authentication is as follows:

In [3]:
# filter warnings or be prepared to see plenty
warnings.filterwarnings('ignore', module='owslib', category=UserWarning)

# connect to web coverage service
wcs = WebCoverageService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS',
                    version='2.0.1',
                    timeout=300)

In [4]:
# print the "abstract" for the GeoMet WCS
print(wcs.identification.abstract)

GeoMet-Weather provides public access to the Meteorological Service of Canada (MSC) and Environment and Climate Change Canada (ECCC) data via interoperable web services and application programming interfaces (API). Through open standards, users can freely and quickly access thousands of real-time and archived weather, climate and water datasets and products and integrate them in their domain-specific applications and decision support systems. Users can build mobile apps, create interactive web maps, and display and animate MSC data in desktop software. MSC GeoMet also enables on demand raw data clipping and reprojection, on demand format conversion and custom visualization.


In [5]:
# some additional wcs metadata
print("URL: " + wcs.url)
print("Version: " + wcs.version)
print("Provider: " + wcs.provider.name)
print("Provider URL: " + wcs.provider.url)

URL: https://geo.weather.gc.ca/geomet?&SERVICE=WCS
Version: 2.0.1
Provider: Government of Canada, Environment and Climate Change Canada, Meteorological Service of Canada
Provider URL: https://eccc-msc.github.io/open-data/msc-geomet/readme_en/


In [6]:
# let's print the operations available from the wcs and the product formatting options
for op in wcs.operations:
    print(op.name, op.formatOptions)

GetCapabilities ['text/xml']
DescribeCoverage ['text/xml']
GetCoverage ['text/xml']


<div class="alert alert-block alert-warning">
    <b>
Note! The Web Coverage Service offers three request types: GetCapabilities gives us information about the web coverage service, DescribeCoverage gives us information about a given layer, and GetCoverage gives us the raw geospatial data for a layer. When we use OWSLib to connect to the web coverage service, we do not need to explicitly make GetCapabilities or DescribeCoverage requests. This is done for us by OWSLib's WCS module.
</div>

## Non-athenticated vs authenticated data

Once we have established a connection to the web coverage service, we can view the available layers.

In [7]:
# check the first 20 contents within it
print(len(wcs.contents))
for key in list(wcs.contents.keys())[0:20]:
    print(key)

6144
CAPS_3km_AbsoluteVorticity_250mb
CAPS_3km_AbsoluteVorticity_500mb
CAPS_3km_AbsoluteVorticity_700mb
CAPS_3km_AbsoluteVorticity_850mb
CAPS_3km_AbsoluteVorticity_1000mb
CAPS_3km_AirDensity_40m
CAPS_3km_AirDensity_80m
CAPS_3km_AirDensity_120m
CAPS_3km_AirTemp_2m
CAPS_3km_AirTemp_40m
CAPS_3km_AirTemp_50mb
CAPS_3km_AirTemp_80m
CAPS_3km_AirTemp_100mb
CAPS_3km_AirTemp_120m
CAPS_3km_AirTemp_150mb
CAPS_3km_AirTemp_175mb
CAPS_3km_AirTemp_200mb
CAPS_3km_AirTemp_225mb
CAPS_3km_AirTemp_250mb
CAPS_3km_AirTemp_275mb


In [8]:
# let's check what data are available for the Water Cycle Prediction System (WCPS)
for key in list(wcs.contents.keys()):
    if "WCPS" in key:
        print(key)

WCPS_1km_AirTemp_1.5m
WCPS_1km_MixedLayerThickness
WCPS_1km_Runoff
WCPS_1km_SeaIceAreaFraction
WCPS_1km_SeaIceCompressiveStrength
WCPS_1km_SeaIceDivergence
WCPS_1km_SeaIceInternalPressure
WCPS_1km_SeaIceShear
WCPS_1km_SeaIceSnowTemp
WCPS_1km_SeaIceSnowVol
WCPS_1km_SeaIceVol
WCPS_1km_SeaSfcHeight
WCPS_1km_SeaWaterPotentialTemp_0.5m
WCPS_1km_SeaWaterSalinity_0.5m
WCPS_1km_TurboclineDepth
WCPS_1km_WindsU_10m
WCPS_1km_WindsV_10m


This list tells us what the layer names are for the available WCPS data, but they aren't very descriptive. It would be nice to look at their full titles instead, but the web coverage service does not store this information. Luckily, we can use the web map service to retrieve this info. 

In [9]:
# let's switch over to the web map service to get a better look at the available data
wms = WebMapService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WMS',
                    version='1.3.0',
                    timeout=300)

# let's have a look at the variable names
for key in list(wms.contents.keys()):
    if "WCPS" in key:
        print(wms[key].title)

Water Cycle Prediction System (WCPS) [1 km]
WCPS-Ocean-Atmosphere footprint
WCPS-Rivers footprint
WCPS-Ocean-Atmosphere
WCPS - Surface air temperature at 1.5 m above ground [K]
WCPS - Ocean mixed layer depth based on density criterion [m]
WCPS - Runoff [mm]
WCPS - Sea ice area fraction
WCPS - Depth-integrated compressive sea ice strength [N/m]
WCPS - Sea ice divergence [%/day]
WCPS - Depth-integrated internal sea ice pressure [N/m]
WCPS - Sea ice shear [%/day]
WCPS - Surface temperature of snow over sea ice or bare sea ice [K]
WCPS - Volume of snow on sea ice per unit grid cell area [m]
WCPS - Sea ice velocity [m/s]
WCPS - Sea ice volume per unit grid cell area [m]
WCPS - Sea surface height above geoid [m]
WCPS - Sea water potential temperature at 0.5 m below the surface [K]
WCPS - Sea water salinity at 0.5 m below the surface [psu]
WCPS - Sea water velocity at 0.5 m below the surface [m/s]
WCPS - Turbocline depth [m]
WCPS - Surface wind U vector at 10 m above ground [m/s]
WCPS - Surfa

<div class="alert alert-block alert-warning">
    <b>
Note! In general, the Web Coverage Service does not allow us to explore the metadata of the available layers without requesting their data. We advise using the Web Map Service if you want to explore the layers and their metadata before making any requests and then switching to the Web Coverage Service when you are ready to pull the data. We discuss this strategy in greater detail below in <a href="#Accessing-layer-information" target="_self">Accessing Layer Information.
</div>

Looking at the variable titles above we can see that nothing is availble for WCPS related to streamflow. Let's switch back to the web coverage service and have a look for DHPS streamflow predictions.

In [10]:
# look for DHPS, nothing returned
for key in list(wcs.contents.keys()):
    if "DHPS" in key:
        print(key)

The reason we don't see any data for DHPS is that the layers are not just password-protected, they are completely hidden.

If we connect to the WCS we can pass a username and password as arguments to the WebCoverageService request. The WebCoverageService does not accept a username and password explicitely. Instead, we must use the Authentication utility.

In [11]:
# pass the layer name to the request
layer_name = 'DHPS_1km_RiverDischarge'

# use auth parameter with Authentication utility
wcs = WebCoverageService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS&COVERAGEID={layer_name}',
                    version='2.0.1',
                    auth=Authentication(username=login['Username'], password=login['Password']),
                    timeout=300)

for key in list(wcs.contents.keys()):
    print(key)

DHPS_1km_RiverDischarge


It is only when we explicitly pass the hidden layer name that we can successfully see the hidden layer in the request contents.

Note again that layers are hidden.  You can see this by connecting to the WCS and providing authentication, but ommitting the layer name from the request:

In [12]:
# use auth parameter with Authentication utility
wcs = WebCoverageService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS',
                    version='2.0.1',
                    auth=Authentication(username=login['Username'], password=login['Password']),
                    timeout=300)

# looking for WCPS, DHPS or EHPS, still no flow predictions are returned
for key in list(wcs.contents.keys()):
    if "WCPS" in key or "DHPS" in key:
        print(key)

WCPS_1km_AirTemp_1.5m
WCPS_1km_MixedLayerThickness
WCPS_1km_Runoff
WCPS_1km_SeaIceAreaFraction
WCPS_1km_SeaIceCompressiveStrength
WCPS_1km_SeaIceDivergence
WCPS_1km_SeaIceInternalPressure
WCPS_1km_SeaIceShear
WCPS_1km_SeaIceSnowTemp
WCPS_1km_SeaIceSnowVol
WCPS_1km_SeaIceVol
WCPS_1km_SeaSfcHeight
WCPS_1km_SeaWaterPotentialTemp_0.5m
WCPS_1km_SeaWaterSalinity_0.5m
WCPS_1km_TurboclineDepth
WCPS_1km_WindsU_10m
WCPS_1km_WindsV_10m


<div class="alert alert-block alert-warning">
<b>So to use the authenticated version of Geomet, whether WMS or WCS, you have to provide the layer name as well as the authentication information, which means you need to know the layer name you're looking for in advance of the request.
</div>

Luckily, here's a list of layers:

In [13]:
# DHPS_1km_CorrectedFlowDir
# DHPS_1km_DeepReservoirStorage_PT24H
# DHPS_1km_DrainageArea
# DHPS_1km_Evap-SpatialAvg12h_PT12H
# DHPS_1km_Precip-SpatialAvg12h_PT12H
# DHPS_1km_RiverChannelStorage_PT24H
# DHPS_1km_RiverDischarge
# DHPS_1km_TotalRunoff-Avg12h_PT12H
# DHPS_1km_WaterbodyID
# DHPS-Analysis_1km_RiverDischarge
# DHPS-Analysis_1km_DeepReservoirStorage
# DHPS-Analysis_1km_Evap-SpatialAvg12h
# DHPS-Analysis_1km_Precip-SpatialAvg12h
# DHPS-Analysis_1km_RiverChannelStorage
# DHPS-Analysis_1km_TotalRunoff-Avg12h

# WCPS-Rivers_1km_CorrectedFlowDir
# WCPS-Rivers_1km_DeepReservoirStorage_PT24H
# WCPS-Rivers_1km_DrainageArea
# WCPS-Rivers_1km_Evap-SpatialAvg12h_PT12H
# WCPS-Rivers_1km_Precip-SpatialAvg12h_PT12H
# WCPS-Rivers_1km_RiverChannelStorage_PT24H
# WCPS-Rivers_1km_RiverDischarge
# WCPS-Rivers_1km_TotalRunoff-Avg12h_PT12H
# WCPS-Rivers_1km_WaterbodyID
# WCPS-Rivers-Analysis_1km_RiverDischarge
# WCPS-Rivers-Analysis_1km_DeepReservoirStorage
# WCPS-Rivers-Analysis_1km_Evap-SpatialAvg12h
# WCPS-Rivers-Analysis_1km_Precip-SpatialAvg12h
# WCPS-Rivers-Analysis_1km_RiverChannelStorage

## Accessing layer information

To know what metadata information comes packaged with a layer, we can look at the layer details.

In [13]:
layer_name = 'DHPS_1km_RiverDischarge'

wcs = WebCoverageService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS&COVERAGEID={layer_name}',
                    version='2.0.1',
                    auth=Authentication(username=login['Username'], password=login['Password']),
                    timeout=300)

# layer details
for key, value in vars(wcs[layer_name]).items():
    if value is not None: # not all layers have values for the full set of available details
        print(key) # let's just print the details applicable to this layer

_elem
_service
id
keywords


<b>
Not much here for metadata! Often we don't know the properties of the data ahead of time. At minimum, we will need to know available datetimes to make a request. To get the layer and time information we need to use the Web Map Service. Once we get the metadata information from the WMS we can come back to the WCS to request the data.

In [14]:
# switching over to the Web Map Service
wms = WebMapService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WMS&LAYERS={layer_name}',
                    version='1.3.0',
                    auth=Authentication(username=login['Username'], password=login['Password']),
                    timeout=300)

# layer details
for key, value in vars(wms[layer_name]).items():
    if value: # not all layers have values for the full set of available details
        print(key) # let's just print the details applicable to this layer

auth
parent
index
id
name
queryable
title
abstract
boundingBoxWGS84
boundingBox
crsOptions
styles
keywords
timepositions
defaulttimeposition
dimensions
metadataUrls


Aha metadata! Let's have a closer look.

In [15]:
# let's have a look at some of the details
print(wms[layer_name].title)
print(wms[layer_name].boundingBoxWGS84)

DHPS - Streamflow discharge [m³/s]
(-168.433366, 38.991633, -52.500045, 71.599969)


Now let's look at the time metadata. For forecasts, there are two types of time metadata, "reference_time" and "time". The reference times refer to the available forecast issue times (model run times). "time" gives information about the timesteps of a specific forecast/analysis issue.

Differently, for analyses, 'reference_time' is not included in the metadata. The "time" metadata gives information on the oldest available analysis time, the most recent available analysis time, and the analysis timestep.

In [16]:
wms[layer_name].dimensions

{'time': {'units': 'ISO8601',
  'default': '2025-07-23T18:00:00Z',
  'nearestValue': '0',
  'values': ['2025-07-23T13:00:00Z/2025-07-29T12:00:00Z/PT1H']},
 'reference_time': {'units': 'ISO8601',
  'default': '2025-07-23T12:00:00Z',
  'multipleValues': '1',
  'nearestValue': '0',
  'values': ['2025-07-21T00:00:00Z/2025-07-23T12:00:00Z/PT12H']}}

In [17]:
# time metadata is stored in the dimensions attribute
wms[layer_name].dimensions['reference_time']['values']

['2025-07-21T00:00:00Z/2025-07-23T12:00:00Z/PT12H']

This is a bit hard to read, let's split the strings and print the time reference time information.

In [18]:
# formatting the reference time information
oldest_fcast, newest_fcast, issue_interval = wms[layer_name].dimensions['reference_time']['values'][0].split('/')
print('The first forecast issue is available at: ' + oldest_fcast)
print('The last forecast issue is available at: ' + newest_fcast)
print('The hourly interval between issued forecasts is: ' + issue_interval)

The first forecast issue is available at: 2025-07-21T00:00:00Z
The last forecast issue is available at: 2025-07-23T12:00:00Z
The hourly interval between issued forecasts is: PT12H


Now we know what forecast issues are available, but we haven't specified a forecast issue or model run in any of our earlier requests. What model run do we receive as a default?

In [19]:
# looking at the default reference time
wms[layer_name].dimensions['reference_time']['default']

'2025-07-23T12:00:00Z'

We can look at the same information for "time". The time information refers to the available datetimes for a given forecast issue or model run.

In [20]:
# formatting the time information
first_datetime, last_datetime, datetime_interval = wms[layer_name].dimensions['time']['values'][0].split('/')
print('The first datetime available is: ' + first_datetime)
print('The last datetime available is: ' + last_datetime)
print('The hourly interval between available times is: ' + datetime_interval)

The first datetime available is: 2025-07-23T13:00:00Z
The last datetime available is: 2025-07-29T12:00:00Z
The hourly interval between available times is: PT1H


The first and last available datetimes above correspond to the first and last available datetimes for the default reference time. There is a default time value as well:

In [21]:
# looking at the default time
wms[layer_name].dimensions['time']['default']

'2025-07-23T18:00:00Z'

## Requesting data with WCS

Now that we know how to access the authenticated streamflow layers and we have the layer metadata information, we have everything we need to make a request. Let's try it out.

In [22]:
# connect using auth / Authentication method 
# (reminder: username and password parameters individually are not accepted by WebCoverageService)
layer_name = 'DHPS_1km_RiverDischarge'

wcs = WebCoverageService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS&COVERAGEID={layer_name}', 
                    auth=Authentication(username=login['Username'], password=login['Password']),
                    version='2.0.1',
                    timeout=300
                    )

First, let's make a request without specifying the time information. 

It is important to note that when requesting a layer from the WCS the grid resolution must be defined. This will ensure the data is not improperly interpolated. Information on resolution can be found in a Describe Coverage request (for example, https://geo.weather.gc.ca/geomet?SERVICE=WCS&VERSION=2.0.1&REQUEST=DescribeCoverage&COVERAGEID=DHPS_1km_RiverDischarge)

When following the above link, look at the offsetVector tags, although the negative sign isn't required when making a `wcs.getCoverage` query:

`<gml:offsetVector srsName="http://www.opengis.net/def/crs/EPSG/0/4326">0 0.008333</gml:offsetVector>`

`<gml:offsetVector srsName="http://www.opengis.net/def/crs/EPSG/0/4326">-0.008333 0</gml:offsetVector>`

In [24]:
response = wcs.getCoverage(identifier = [layer_name], 
                    format = 'image/netcdf', 
                    subsettingcrs = 'EPSG:4326', # default and recommended crs
                    subsets = [('lat', 45.0, 58.0), ('lon', -70.0, -52.0)],
                    resolutions=[('lat', 0.008333), ('lon', 0.008333)]
                   )

Now let's have a look at the data "under-the-hood" of the response. We'll do this by loading the data into an xarray <i>DataArray</i>. 

In [25]:
# read into an xarray
ds = xr.open_dataset(response.read(), engine='scipy').load()
# print the header for the DataArray
ds.head

<bound method Dataset.head of <xarray.Dataset> Size: 13MB
Dimensions:         (lat: 1560, lon: 2100)
Coordinates:
  * lat             (lat) float64 12kB 45.0 45.01 45.02 ... 57.98 57.99 58.0
  * lon             (lon) float64 17kB -70.0 -69.99 -69.98 ... -52.51 -52.5
Data variables:
    crs             |S1 1B b''
    RiverDischarge  (lat, lon) float32 13MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
Attributes:
    Conventions:                  CF-1.8
    institution:                  Canadian Centre for Meteorological and Envi...
    product_version:              DHPS version v3.6.0
    source:                       Deterministic_Hydrological_Prediction_Syste...
    GDAL_TIFFTAG_RESOLUTIONUNIT:  2
    GDAL_TIFFTAG_XRESOLUTION:     72.0
    GDAL_TIFFTAG_YRESOLUTION:     72.0
    GDAL:                         GDAL 3.6.4, released 2023/04/17
    history:                      Wed Jul 23 18:06:23 2025: GDAL CreateCopy( ...>

The DataArray's header gives us a quick glance at the layer and its geospatial information. Typically, the <i>Dimensions</i> describe each axis of the data (x, y, and in some cases, z). The <i>Coordinates</i> give us the latitude and longitude arrays for the layer that we could use to construct the layer's grid. The <i>Data variables</i> are listed specific to the requested layer. The <i>Attributes</i> describe the version of GDAL used to create the data. 

<b>Note that the data from the response does not contain any time information. We made the request without specifying a time, so this data must be associated with the default time and default reference time. If we hadn't used the WMS earlier to look at the default time positions, we wouldn't know what time positions to attribute to this data. Let's see if we can find the time information from the WCS response. First, Let's check the url associated with the request.

In [26]:
# note the lack of time info
response.geturl()

'https://geo.weather.gc.ca/geomet?version=2.0.1&request=GetCoverage&service=WCS&CoverageID=DHPS_1km_RiverDischarge&format=image/netcdf&subsettingcrs=EPSG:4326&subset=lat(45.0,58.0)&subset=lon(-70.0,-52.0)&resolution=lat(0.008333)&resolution=lon(0.008333)'

The time info isn't obvious from the request url. Let's check the response info instead.

In [27]:
response.info()

{'Date': 'Wed, 23 Jul 2025 18:06:22 GMT', 'Server': 'Apache', 'Strict-Transport-Security': 'max-age=63072000; preload, max-age=63072000; preload', 'Content-Disposition': 'attachment; filename="geomet-weather_DHPS_1km_RiverDischarge_2025-07-23T1200Z_PT6H.nc"', 'Access-Control-Allow-Origin': '*', 'Access-Control-Allow-Headers': 'Origin, X-Requested-With, Content-Type, Accept, Authorization', 'Access-Control-Allow-Methods': 'POST, GET, OPTIONS', 'Content-Type': 'image/netcdf', 'Keep-Alive': 'timeout=5, max=100', 'Connection': 'Keep-Alive', 'Transfer-Encoding': 'chunked'}

In [28]:
response.info()['Content-Disposition']

'attachment; filename="geomet-weather_DHPS_1km_RiverDischarge_2025-07-23T1200Z_PT6H.nc"'

The filename stored in the response information gives us an idea of the time position for this data. The YYYY-MM-DDT<i>xxxx</i>Z is the datetime that the forecast was issued and PT<i>xx</i>H is the forecast hour.

<div class="alert alert-block alert-warning">
<b>Finding the time positions from a default WCS request is not intuitive and can only be done indirectly. For this reason, we do not advise making a request from the WCS without specfiying the time and reference time information. The time metadata is available with the WMS. For more information please read the preceeding section, <a href="#Accessing-layer-information" target="_self">Accessing Layer Information. 
</div>

Let's make the request again, but this time let's use the time positions we found with the WMS. We will request the first available forecast time from the oldest forecast available.

In [29]:
oldest_fcast # issue time of the oldest forecast available

'2025-07-21T00:00:00Z'

In [30]:
datetime_interval # interval of available datetimes

'PT1H'

In [31]:
# convert the datetime string to a datetime object
iso_format = "%Y-%m-%dT%H:%M:%SZ"
oldestf = datetime.strptime(oldest_fcast, iso_format)
oldestf

datetime.datetime(2025, 7, 21, 0, 0)

In [32]:
# remove the non-numerical characters from the datetime interval and convert to an integer
intvl = int(re.sub(r'\D', '', datetime_interval))
intvl

1

In [33]:
# find the datetime object for the first available forecast hour
firsthr = oldestf + timedelta(hours=intvl)
# convert back to a string for the request
first_fcasthr = datetime.strftime(firsthr, iso_format)
first_fcasthr # first available time from the most recent forecast

'2025-07-21T01:00:00Z'

Note that for getting the response below, DIM_REFERENCE_TIME is defined. However, if querying an analyses, this parameter should not be used.

In [34]:
# redo the response with the DIM_REFERENCE_TIME and TIME arguments
response = wcs.getCoverage(identifier = [layer_name], 
                    format = 'image/netcdf', 
                    subsettingcrs = 'EPSG:4326', 
                    subsets = [('lat', 48.0, 52.0), ('lon', -98.0, -92.0)],
                    resolutions=[('lat', 0.008333), ('lon', 0.008333)], 
                    DIM_REFERENCE_TIME=oldest_fcast, # capitalization here is important
                    TIME=first_fcasthr # capitalization here is important
                   )

If the time and reference time were not defined above, it would use the defaults of the first timestep of most recent forecast/analysis.

<div class="alert alert-block alert-warning">
<b>Note! When we specified the time and reference time in the above request we capitalized the time and reference time arguments. Without capitalization, the time positions do not get passed to the GetCoverage request and the data from the default time positions is returned instead. This is a suspected issue with the OWSLib WCS module.
</div>

Let's take a peak at the filename again now that we've requested a different datetime ...

In [35]:
response.info()['Content-Disposition']

'attachment; filename="geomet-weather_DHPS_1km_RiverDischarge_2025-07-21T0000Z_PT1H.nc"'

Looks like we got the time we requested. Let's load the data into a DataArray and add this time information.

In [36]:
# read into an xarray
ds = xr.open_dataset(response.read()).load()
# add the time metadata as a new dimension and coordinate
ds = ds.expand_dims(time=[firsthr])
# print the header for the DataArray, note the new time information
ds.head

<bound method Dataset.head of <xarray.Dataset> Size: 1MB
Dimensions:         (time: 1, lat: 480, lon: 720)
Coordinates:
  * time            (time) datetime64[ns] 8B 2025-07-21T01:00:00
  * lat             (lat) float64 4kB 48.0 48.01 48.02 ... 51.98 51.99 52.0
  * lon             (lon) float64 6kB -98.0 -97.99 -97.98 ... -92.01 -92.0
Data variables:
    crs             (time) |S1 1B b''
    RiverDischarge  (time, lat, lon) float32 1MB 0.002456 0.004243 ... 0.0 0.0
Attributes:
    Conventions:                  CF-1.8
    institution:                  Canadian Centre for Meteorological and Envi...
    product_version:              DHPS version v3.6.0
    source:                       Deterministic_Hydrological_Prediction_Syste...
    GDAL_TIFFTAG_RESOLUTIONUNIT:  2
    GDAL_TIFFTAG_XRESOLUTION:     72.0
    GDAL_TIFFTAG_YRESOLUTION:     72.0
    GDAL:                         GDAL 3.6.4, released 2023/04/17
    history:                      Wed Jul 23 18:07:11 2025: GDAL CreateCopy( ...>

In [37]:
# add the reference time as an attribute of  RiverDischarge
ds.RiverDischarge.attrs['reference_time'] = oldest_fcast
ds.RiverDischarge.attrs

{'long_name': 'Mean discharge value exiting the river channel over the hour ending at the indicated time',
 'shortname': 'Streamflow discharge',
 'units': 'm**3/s',
 'grid_mapping': 'crs',
 'reference_time': '2025-07-21T00:00:00Z'}

Et voila! 

## Saving requested data to netCDF or geoJSON

Saving requested data to a netCDF file is very easy with xarray. Let's save the dataset we requested earlier.

In [38]:
# save to netcdf
ds.to_netcdf(path=layer_name+'_test_data.nc')

Saving the data to geoJSON is also simple, but takes a few more steps.

In [39]:
# convert the DataArray to a pandas DataFrame
df = ds.to_dataframe()
# remove crs from dataframe to work with RiverDischarge properly
df = df.drop('crs', axis=1) 
df.head()

RiverDischarge
time                lat       lon                       
2025-07-21 01:00:00 48.004166 -97.995834        0.002456
                              -97.987500        0.004243
                              -97.979167        0.005436
                              -97.970833        0.000000
                              -97.962500        0.000056

In [40]:
df = df.reset_index()
df.head()

,time,lat,lon,RiverDischarge
0,2025-07-21 01:00:00,48.004166,-97.995834,0.002456
1,2025-07-21 01:00:00,48.004166,-97.987500,0.004243
2,2025-07-21 01:00:00,48.004166,-97.979167,0.005436
3,2025-07-21 01:00:00,48.004166,-97.970833,0.000000
4,2025-07-21 01:00:00,48.004166,-97.962500,0.000056


In [41]:
# bundle the latitudes and longitude in Point geometry tuples
geom = [Point(x,y) for x, y in zip(df['lon'], df['lat'])]

In [42]:
# transform the DataFrame into a GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=geom, crs='EPSG:4326')
gdf.head()

,time,lat,lon,RiverDischarge,geometry
0,2025-07-21 01:00:00,48.004166,-97.995834,0.002456,POINT (-97.99583 48.00417)
1,2025-07-21 01:00:00,48.004166,-97.987500,0.004243,POINT (-97.9875 48.00417)
2,2025-07-21 01:00:00,48.004166,-97.979167,0.005436,POINT (-97.97917 48.00417)
3,2025-07-21 01:00:00,48.004166,-97.970833,0.000000,POINT (-97.97083 48.00417)
4,2025-07-21 01:00:00,48.004166,-97.962500,0.000056,POINT (-97.9625 48.00417)


In [43]:
# remove the unneeded latitude and longitude series
gdf = gdf.drop(['lat', 'lon'], axis=1)
gdf.head()

,time,RiverDischarge,geometry
0,2025-07-21 01:00:00,0.002456,POINT (-97.99583 48.00417)
1,2025-07-21 01:00:00,0.004243,POINT (-97.9875 48.00417)
2,2025-07-21 01:00:00,0.005436,POINT (-97.97917 48.00417)
3,2025-07-21 01:00:00,0.000000,POINT (-97.97083 48.00417)
4,2025-07-21 01:00:00,0.000056,POINT (-97.9625 48.00417)


In [44]:
# save to json
gdf.to_file(layer_name+'_test_data.json', driver='GeoJSON')

## Bonus: Visualizing requested data with hvplot

In this example we will put together everything we reviewed above for the WCS by exploring, requesting, loading, and visualizing streamflow predictions from DHPS. For this part of the tutorial we are going to use [hvplot](https://hvplot.holoviz.org/), an interactive visualization API. 

Our goal here is to create an animated map of streamflows from the latest forecast issue. To do this we will need to make a WCS request for each forecast time in the forecast issue. As such, we first need to retrieve the time metadata for the DHPS streamflow forecasts. 

In [45]:
# loading login information
config = configparser.ConfigParser()
config.read_file(open('config.cfg'))

login = config['Login']

layer_name = 'DHPS_1km_RiverDischarge'

# first querying the WMS for time metadata
wms = WebMapService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WMS&LAYERS={layer_name}',
                    version='1.3.0',
                    auth=Authentication(username=login['Username'], password=login['Password']),
                    timeout=300)

first_datetime, last_datetime, datetime_interval = wms[layer_name].dimensions['time']['values'][0].split('/')
oldest_fcast, newest_fcast, issue_interval = wms[layer_name].dimensions['reference_time']['values'][0].split('/')

iso_format = "%Y-%m-%dT%H:%M:%SZ"

# convert dates to datetime objects
first = datetime.strptime(first_datetime, iso_format)
last = datetime.strptime(last_datetime, iso_format)

# remove anything that isn't a number from the datetime interval (time between forecasts)
intvl = int(re.sub(r'\D', '', datetime_interval))

# create a list of forecast datetimes (we will add these to the requested data)
fcasthrs = [first]
while first < last:
    first = first + timedelta(hours=intvl)
    fcasthrs.append(first)

# create a list of iso formatted forecast datetime strings (we will use these in the WCS requests)
fcasthrs_str = [datetime.strftime(hr, iso_format) for hr in fcasthrs]

Note: the next cell may take a few minutes to run because several requests are being made.

In [46]:
wcs = WebCoverageService(f'https://geo.weather.gc.ca/geomet?&SERVICE=WCS&COVERAGEID={layer_name}', 
                    auth=Authentication(username=login['Username'], password=login['Password']),
                    version='2.0.1',
                    timeout=300
                    )

# for each forecast hour, make a WCS request
arrys = []

for i, hr in enumerate(fcasthrs_str):
    response = wcs.getCoverage(identifier = [layer_name], 
                        format = 'image/netcdf', 
                        subsettingcrs = 'EPSG:4326', 
                        subsets = [('lat', 46.0, 52.0), ('lon', -96.0, -90.0)],
                        resolutions=[('lat', 0.008333), ('lon', 0.008333)],
                        DIM_REFERENCE_TIME=newest_fcast, 
                        TIME=hr 
                       )
    
    # read into an xarray
    ds = xr.open_dataset(response.read()).load()
    
    # add the time metadata as a new dimension and coordinate
    ds = ds.expand_dims(time=[fcasthrs[i]])
    
    # append to list of xarrays
    arrys.append(ds)
    
fcasts = xr.concat([ds for ds in arrys], dim='time')
# remove crs data variable to work with RiverDischarge properly
fcasts = fcasts.drop_vars('crs')

In [47]:
# checking to see that the time and lat/lon dimensions look sensible
fcasts.head()

<xarray.Dataset> Size: 620B
Dimensions:         (time: 5, lat: 5, lon: 5)
Coordinates:
  * time            (time) datetime64[ns] 40B 2025-07-23T13:00:00 ... 2025-07...
  * lat             (lat) float64 40B 46.0 46.01 46.02 46.03 46.04
  * lon             (lon) float64 40B -96.0 -95.99 -95.98 -95.97 -95.96
Data variables:
    RiverDischarge  (time, lat, lon) float32 500B 0.002937 0.01242 ... 0.01711
Attributes:
    Conventions:                  CF-1.8
    institution:                  Canadian Centre for Meteorological and Envi...
    product_version:              DHPS version v3.6.0
    source:                       Deterministic_Hydrological_Prediction_Syste...
    GDAL_TIFFTAG_RESOLUTIONUNIT:  2
    GDAL_TIFFTAG_XRESOLUTION:     72.0
    GDAL_TIFFTAG_YRESOLUTION:     72.0
    GDAL:                         GDAL 3.6.4, released 2023/04/17
    history:                      Wed Jul 23 18:09:10 2025: GDAL CreateCopy( ...

The data looks good! Now we can create a plot.

In [48]:
# remove crs data variable to work with RiverDischarge properly
ds = ds.drop_vars('crs')
# hvplot can be used to show an interactive map of the xarray (tip: hover your cursor over the plot)
ds.sel(time=first).hvplot(width=800, height=700, cmap='viridis', cnorm='linear')

:Image   [lon,lat]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)

The plot looks good, but we can't see much because the flows in the larger rivers saturate the colormap. hvplot gives us a couple options to normalize the colormap. In the plot above we specified that the colormap should be linear (cnorm='linear'). This time, let's try using a [historgram equalization](https://en.wikipedia.org/wiki/Histogram_equalization) to normalize the colormap.   

In [49]:
ds.sel(time=first).hvplot(width=800, height=700, cmap='viridis', cnorm='eq_hist')

:Image   [lon,lat]   (Mean discharge value exiting the river channel over the hour ending at the indicated time)

Much better! The two images above are just showing us the first forecast time. Let's animate these images to look at how the streamflow is expected to change spatially and temporally during the forecast.

In [51]:
# depending on your environment and package versions, you may notice several Shapely Deprecation Warnings
# they don't impact our ability to create these plots, so let's suppress them for now.
from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning) 

# create an animation of the forecast, notice here that we are projecting the data
time = pnw.Player(name='time', start=0, end=len(fcasthrs), loop_policy='once', interval=20)
fcasts.interactive(loc='bottom').isel(time=time).hvplot(width=800, 
                                                        height=800, 
                                                        cmap='viridis', 
                                                        cnorm='eq_hist',
                                                        projection=ccrs.PlateCarree(), 
                                                        geo=True)

BokehModel(combine_events=True, render_bundle={'docs_json': {'04fee49a-4184-48ad-81cb-4873594aeff5': {'version…

Cool!